# 使用するライブラリの読み込み

In [1]:
import datetime
import glob
from io import StringIO
#import os
import random
import re
import time
import pickle

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions
from selenium.webdriver.chrome.options import Options

import chromedriver_binary

import chainer
import chainer.functions as F
import chainer.links as L
from chainer import serializers

import my_snipets

# 出馬表のデータ作成

## HTMLの取得

### 各種関数の定義

In [2]:
def makeRaceDateURLList(date:datetime.date) -> str:
    """
    対象日付の開催レース一覧ページのURLを生成する関数
    `date`: 対象の年月日 (datetime.dateオブジェクト)
    """
  
    #対象年月の開催カレンダーのURLを生成
    url = f"https://race.netkeiba.com/top/calendar.html?year={date.year}&month{date.month}"

    #カレンダーページのHTMLを取得
    headers = {'User-Agent': random.choice(my_snipets.makeUserAgents())}
    html = requests.get(url, headers=headers)
    html.encoding = "EUC-JP"

    #HTMLをsoupオブジェクトに変換
    soup = BeautifulSoup(html.text)

    #開催日の一覧を取得し、開催日ページのURLを取得
    RaceCellBoxAll = soup.find_all("td",attrs={"class":"RaceCellBox"})
    
    for i in range(len(RaceCellBoxAll)):
        if RaceCellBoxAll[i].find("a") is not None:
            day = RaceCellBoxAll[i].find("span",attrs={"class":"Day"}).text
            if datetime.date(date.year, date.month, int(day)) == date:
                returnStr = (RaceCellBoxAll[i].find("a")["href"].replace("..","https://race.netkeiba.com/"))
                break

    return returnStr

In [3]:
def makeRaceURLList(tgtURL:str) -> pd.DataFrame:
    """
    開催日ページのURLから、各レースのURLリストと発走日時のデータフレーム(発走日時昇順)を作成する関数  
    `tgtURL`: 対象の開催日ページのURL (文字列)
    """
    returnDF = pd.DataFrame()
    URLList = []
    datetimeList = []
    dateStr = tgtURL.split("=")[1]

    # Seleniumを使用してブラウザを起動し、対象URLにアクセス
    option = Options()
    option.add_argument('--headless')
    browser = webdriver.Chrome(options=option)
    browser.get(tgtURL)

    # ブラウザから必要な情報を取得
    RaceListDataItemAll = browser.find_elements(By.CLASS_NAME,"RaceList_DataItem")
    for item in RaceListDataItemAll:
        if len(item.find_element(By.CLASS_NAME,"RaceData").find_element(By.TAG_NAME,"span").text) > 0:
            timeStr = item.find_element(By.CLASS_NAME,"RaceData").find_element(By.TAG_NAME,"span").text
        else:
            timeStr = "23:59"

        URLList.append(item.find_element(By.TAG_NAME,"a").get_attribute("href"))
        datetimeList.append(datetime.datetime(int(dateStr[0:4]), int(dateStr[4:6]), int(dateStr[6:8]),int(timeStr[0:2]), int(timeStr[3:5])))

    # ブラウザを閉じる
    browser.quit()
    
    # データフレームに変換
    returnDF = pd.DataFrame({"URL": URLList,"datetime": datetimeList})
    returnDF.sort_values(by="datetime", inplace=True)
    returnDF.reset_index(drop=True, inplace=True)

    print(f"対象日付: {dateStr}" + "\n" + f"レース数: {len(returnDF)}")

    return returnDF

In [4]:
def screpingRaceCardHTML(raceURL:str):
    """
    レースカードのHTMLをスクレイピングする関数
    `RaceURL`: レースカードページのURL
    """
    raceID = re.sub(r"\D+", "", str(raceURL.split("/")[-1:]))
    option = Options()
    option.add_argument('--headless')
    browser = webdriver.Chrome(options=option)
    browser.get(raceURL)
    html = browser.page_source
    with open(f"../data/html/raceCard/{raceID}.txt","w") as f:
            f.write(html)
    browser.quit()

    print(f"[raceID:{raceID}]出馬表のHTMLをスクレイピングしました。")

    return raceID

### 実行

In [78]:
tgtyear = 2025
tgtmonth = 6
tgtday = 21

In [79]:
tgtdate = datetime.date(tgtyear, tgtmonth, tgtday)  # 対象日付を指定
raceListURL = makeRaceDateURLList(tgtdate)
raceALLDF = makeRaceURLList(raceListURL)
raceID = screpingRaceCardHTML(raceALLDF["URL"][0])

対象日付: 20250621
レース数: 36
[raceID:202502010301]出馬表のHTMLをスクレイピングしました。


## データ作成

### データ作成用関数の定義

In [5]:
def makeRaceCardDataFrame(tgtRaceID):
    """
    レースカードのHTMLからDataFrameを作成する関数
    `raceID`: レースID
    """
    with open(f"../data/html/raceCard/{tgtRaceID}.txt", "r") as f:
        html = f.read()

    returnDF = pd.read_html(StringIO(html))[0]

    # raceCardDFのカラム名を整形
    columnList = returnDF.columns.get_level_values(1)
    returnDF.columns = columnList
    
    # HTMLをsoupオブジェクトに変換
    soup = BeautifulSoup(html)

    # raceCardDFにraceID/horseID/jockeyIDを追加
    horseIDList = []
    jockeyIDList = []
    HorseListAll = soup.find_all("tr",attrs={"class":"HorseList"})
    cnt = 0
    while cnt < len(returnDF):
        horseID = HorseListAll[cnt].find("span",attrs={"class":"HorseName"}).find("a")["href"].split("/")[-1]
        if HorseListAll[cnt].find("td",attrs={"class":"Jockey"}).find("a") != None:
            jockeyID = HorseListAll[cnt].find("td",attrs={"class":"Jockey"}).find("a")["href"].split("/")[-2]
        else:
            jockeyID = "00000"
        horseIDList.append(horseID)
        jockeyIDList.append(jockeyID)
        cnt += 1

    returnDF["raceID"] = tgtRaceID
    returnDF["horseID"] = horseIDList
    returnDF["jockeyID"] = jockeyIDList

    # レース名
    raceName = soup.find("h1", attrs={"class": "RaceName"}).text.strip()

    # レース情報
    raceInfoList = soup.find("div", attrs={"class": "RaceData01"}).text.split("/")
    
    #距離
    course_len = re.findall("\d+",raceInfoList[1].strip())[0]

    #レースタイプ
    if raceInfoList[1].strip().find("障") != -1:
        raceType = "障害"
    elif raceInfoList[1].strip().find("芝") != -1:
        raceType = "芝"
    elif raceInfoList[1].strip().find("ダ") != -1:
        raceType = "ダート"

    #天候
    try:
        weather = raceInfoList[2].strip().split(":")[1]
    except IndexError:
        weather = np.nan

    #馬場状態
    try:
        groundState = raceInfoList[3].strip().split(":")[1]
    except IndexError:
        groundState = np.nan

    returnDF["raceName"] = raceName
    returnDF["course_len"] = course_len
    returnDF["weather"] = weather
    returnDF["raceType"] = raceType
    returnDF["groundState"] = groundState
    returnDF["date"] =  pd.to_datetime(f"{tgtyear}年{tgtmonth}月{tgtday}日",format="%Y年%m月%d日")

    return returnDF

### 実行

In [10]:
raceCardDF = makeRaceCardDataFrame(raceID)

In [11]:
pd.set_option('display.max_columns', raceCardDF.shape[1])
raceCardDF

,枠,馬 番,印,馬名,性齢,斤量,騎手,厩舎,馬体重 (増減),予想 オッズ,人気,登録,グループ,馬メモ切替,raceID,horseID,jockeyID,raceName,course_len,weather,raceType,groundState,date
0,NaN,NaN,--,アドバンスオーセン,牡3,54.0,▲川端,栗東森秀,NaN,115.8,15,NaN,NaN,編集,202502010301,2022110086,01195,3歳未勝利,1200,NaN,芝,NaN,2025-06-21
1,NaN,NaN,--,グッバイウェーブ,牝3,55.0,横山武,美浦中川,NaN,8.8,5,NaN,NaN,編集,202502010301,2022102492,01170,3歳未勝利,1200,NaN,芝,NaN,2025-06-21
2,NaN,NaN,--,グリフォン,牝3,55.0,鮫島駿,栗東小椋,NaN,5.0,2,NaN,NaN,編集,202502010301,2022103899,01157,3歳未勝利,1200,NaN,芝,NaN,2025-06-21
3,NaN,NaN,--,クレドール,牝3,55.0,原田和,美浦小笠,NaN,64.1,12,NaN,NaN,編集,202502010301,2022105653,01143,3歳未勝利,1200,NaN,芝,NaN,2025-06-21
4,NaN,NaN,--,ゴールドサニーデイ,牝3,55.0,藤岡佑,栗東渡辺,NaN,10.2,6,NaN,NaN,編集,202502010301,2022105330,01093,3歳未勝利,1200,NaN,芝,NaN,2025-06-21
5,NaN,NaN,--,シルフラ,牝3,55.0,佐々木,美浦大竹,NaN,13.9,8,NaN,NaN,編集,202502010301,2022107077,01197,3歳未勝利,1200,NaN,芝,NaN,2025-06-21
6,NaN,NaN,--,デルタニュートラル,牝3,52.0,▲上里,美浦加藤士,NaN,24.0,10,NaN,NaN,編集,202502010301,2022107243,01217,3歳未勝利,1200,NaN,芝,NaN,2025-06-21
7,NaN,NaN,--,ニシノコイブミ,牝3,55.0,武豊,美浦西田,NaN,4.5,1,NaN,NaN,編集,202502010301,2022100400,00666,3歳未勝利,1200,NaN,芝,NaN,2025-06-21
8,NaN,NaN,--,ファリア,牡3,54.0,▲古川奈,美浦土田,NaN,81.8,13,NaN,NaN,編集,202502010301,2022106894,01190,3歳未勝利,1200,NaN,芝,NaN,2025-06-21
9,NaN,NaN,--,フィリグラン,牝3,55.0,原,栗東森田,NaN,117.1,16,NaN,NaN,編集,202502010301,2022100644,01184,3歳未勝利,1200,NaN,芝,NaN,2025-06-21


# 出馬する馬の情報を取得

In [12]:
#raceCardDFのhorseIDのデータを作成
from makeRawData import makeHorseData
a = makeHorseData(raceCardDF["horseID"].tolist())
a.getHorseHTML(skipResult=False)
a.makeHorseRawData()

現在 [horseID:2022101412] の血統情報データを作成中(16/16)/16)

# スコアリング用データ作成

In [11]:
class preprocessing:
    """
    出馬表のデータの前処理を実施し、スコアリング用データを作成するクラス
    """
    def __init__(self, df: pd.DataFrame):
        self.df = df

    def raceCardPreprocess(self) -> None:
        """
        出馬表データに対して、前処理を実行するメソッド
        """
        # カラム名の整形
        newColumns = []
        for col in self.df.columns:
            newColumns.append(col.replace(" ",""))
        self.df.columns = newColumns

        #性齢 → Sex/Age
        self.df["Sex"] = self.df["性齢"].astype(str).str[:1]
        self.df["Age"] = self.df["性齢"].str.extract(r'(\d+)').astype(int)

        #馬体重(増減) → Weight/WeightVariation
        try:
            self.df["Weight"] = self.df["馬体重(増減)"].str.split("(",expand=True)[0].astype(int)
            self.df["WeightVariation"] = self.df["馬体重(増減)"].str.split("(",expand=True)[1].str[:-1].astype(int)
        except AttributeError:
            self.df["Weight"] = np.nan
            self.df["WeightVariation"] = np.nan

        #カラムの変数型の成型
        self.df["course_len"] = self.df["course_len"].astype(int)

        return

    def makedataHorseResults(self) -> None:
        """
        出馬表に存在する全ての馬の過去成績のデータを縦結合する
        """

        self.horseResults = pd.DataFrame()
        horseIDList = self.df["horseID"].tolist()
        cnt = 1
        for horseID in horseIDList:
            print("\r" + "(" + str(cnt) + "/" + str(len(horseIDList)) + ")",end="")

            dftmp = pd.read_pickle(f"../data/rawData/horse/{horseID}.pickle")

            indexdf = []
            for i in range(len(dftmp)):
                indexdf.append(horseID)

            dftmp.index = indexdf
            self.horseResults = pd.concat([self.horseResults,dftmp])
            cnt += 1
        
        # カラム名の整形
        newColumns = []
        for col in self.horseResults.columns:
            newColumns.append(col.replace(" ",""))
        self.horseResults.columns = newColumns

        return
    
    def horseResultsPreprocess(self) -> None:
        """
        過去成績のデータを前処理する関数
        """
        self.horseResults["Rank_y"] = self.horseResults["着順"].astype(str).str.strip("()降再")
        self.horseResults = self.horseResults[self.horseResults["Rank_y"].astype(str).str.contains("\d")]
        self.horseResults["Rank_y"] = self.horseResults["Rank_y"].astype(float)
        self.horseResults["Rank_Y"] = self.horseResults["Rank_y"].astype(int)

        #変数の型を修正
        self.horseResults["date"] = pd.to_datetime(self.horseResults["日付"])

        #不要変数の削除
        self.horseResults.drop(columns=["着順","日付"], inplace=True)

        return
    
    def makeVarFromHorseResults(self) -> None:
        """
        馬の過去成績から変数を作成し、出馬表データに付与し、スコアリング用データを作成するメソッド
        作成変数===================  
        pre1Rank  :前1走の順位  
        pre2Rank  :前2走の順位  
        pre3Rank  :前3走の順位  
        pre4Rank  :前4走の順位  
        pre5Rank  :前5走の順位  
        preAllRank:過去すべての順位の平均  
        pre1Term  :前1走から今回までの期間(日)  
        pre2Term  :前2走から前1走までの期間(日)  
        pre3Term  :前3走から前2走までの期間(日)  
        pre4Term  :前4走から前3走までの期間(日)  
        pre5Term  :前5走から前4走までの期間(日)
        """

        self.forScoringDF = self.df.copy()
        #過去の順位系の変数作成===================================================================================================
        #pre1Rank,pre2Rank,pre3Rank,preAllRankMean
        #馬結果テーブルを中央の結果のみにする
        forPreRank = self.horseResults[self.horseResults['開催'].str.contains("札幌|函館|福島|新潟|東京|中山|中京|京都|阪神|小倉")]

        #レース結果テーブルと馬結果テーブルを結合(Rank,date,)
        forPreRank2 = self.df.merge(forPreRank[["Rank_y","date"]], left_on = "horseID", right_index = True, how = "left")

        #対象レースより過去の日付の結果のみに絞る
        forPreRank2 = forPreRank2[forPreRank2["date_x"] > forPreRank2["date_y"]]
        #レースID,horseIDをキーに連番をふる(何個前のレースかの番号)
        forPreRank2["no"] = forPreRank2.groupby(["raceID","horseID"]).cumcount()

        #過去レースの順位テーブルを作成
        for i in range(5):
            preRankTmp = pd.DataFrame()
            preRankTmp = forPreRank2[forPreRank2["no"] == i][["raceID","horseID","Rank_y"]]
            preRankTmp.rename(columns={'Rank_y': f'pre{str(i + 1)}Rank'},inplace=True)
            self.forScoringDF = self.forScoringDF.merge(preRankTmp, left_on=["raceID","horseID"],right_on=["raceID","horseID"],how="left")

        preAllRankMean = forPreRank2.groupby(["raceID","horseID"]).mean("Rank_y")
        preAllRankMean.rename(columns={"Rank_y":"preAllRankMean"},inplace=True)
        self.forScoringDF = self.forScoringDF.merge(preAllRankMean["preAllRankMean"],left_on=["raceID","horseID"],right_index=True,how="left")
        #過去実績のない馬に対する処理を追加する必要あり

        #過去のレース間日数系の変数作成============================================================================================
        #pre1Term,pre2Term,pre3Term
        #レース結果テーブルと馬結果テーブルを結合(Rank,date,)
        forPreTerm = self.df.merge(self.horseResults[["Rank_y","date"]], left_on = "horseID", right_index = True, how = "left")
        #対象レースより過去の日付の結果のみに絞る
        forPreTerm = forPreTerm[forPreTerm["date_x"] > forPreTerm["date_y"]]
        #レースID,horseIDをキーに連番をふる(何個前のレースかの番号)
        forPreTerm["no"] = forPreTerm.groupby(["raceID","horseID"]).cumcount()

        #過去のレース間の日数を算出
        preTermFin = pd.DataFrame()
        termColumnsList = ["raceID","horseID"]
        for i in range(5):
            preTermTmp = forPreTerm[forPreTerm["no"] == i][["raceID","horseID","date_x","date_y"]]
            preTermTmp.rename(columns={"date_y":f"pre{str(i + 1)}date"},inplace=True)

            if i == 0 :
                preTermFin = preTermTmp
                preTermFin.rename(columns={"date_x":"pre0date"},inplace=True)
            else:
                preTermFin = preTermFin.merge(preTermTmp[["raceID","horseID",f"pre{str(i + 1)}date"]],left_on=["raceID","horseID"],right_on=["raceID","horseID"],how="left")

            preTermFin[f"pre{str(i + 1)}Term"] = preTermFin[f"pre{str(i)}date"] - preTermFin[f"pre{str(i + 1)}date"]
            preTermFin[f"pre{str(i + 1)}Term"] = preTermFin[f"pre{str(i + 1)}Term"] / datetime.timedelta(days=1)
            termColumnsList.append(f"pre{str(i + 1)}Term")

        #dftmpにレース間日数情報を結合
        self.forScoringDF = self.forScoringDF.merge(preTermFin[termColumnsList],left_on=["raceID","horseID"],right_on=["raceID","horseID"],how="left")

        return
    
    def forScoringDFPreprocess(self) -> pd.DataFrame:
        """
        スコアリング用データの前処理をするメソッド
        """
        #カラム名の成形
        self.forScoringDF = self.forScoringDF.rename(columns={"枠":"枠番","オッズ更新":"単勝"})

        #必要なカラムだけ残す
        self.forScoringDF = self.forScoringDF[[
            '枠番',
            '馬番', 
            '斤量',
            '単勝',
            '人気',
            'course_len',
            'weather',
            'raceType',
            'groundState',
            'Sex',
            'Age',
            'Weight',
            'WeightVariation',
            'pre1Rank',
            'pre2Rank',
            'pre3Rank',
            'pre4Rank',
            'pre5Rank',
            'preAllRankMean',
            'pre1Term',
            'pre2Term',
            'pre3Term',
            'pre4Term',
            'pre5Term']]
        
        #labelEncoding
        categorical_features = ["weather","raceType","groundState","Sex"]
        for col in categorical_features:
            with open(f"../data/labelencoder/{col}_labelEncoder.pickle","rb") as f:
                LE = pickle.load(f)

            LE.classes_ = np.append(LE.classes_,np.nan)
            self.forScoringDF[col] = LE.transform(self.forScoringDF[col])
            self.forScoringDF[col] = self.forScoringDF[col].astype("category")
        
        #欠損値補完
        self.forScoringDF = self.forScoringDF.fillna({
            "枠番":-1,
            "馬番":-1,
            "Weight":-1,
            "WeightVariation":-1,
            "pre1Rank":-1,
            "pre2Rank":-1,
            "pre3Rank":-1,
            "pre4Rank":-1,
            "pre5Rank":-1,
            "preAllRankMean":-1,
            "pre1Term":-1,
            "pre2Term":-1,
            "pre3Term":-1,
            "pre4Term":-1,
            "pre5Term":-1})
        
        return self.forScoringDF

In [14]:
a = preprocessing(raceCardDF)
a.raceCardPreprocess()
a.makedataHorseResults()
a.horseResultsPreprocess()
a.makeVarFromHorseResults()
forScoringDF = a.forScoringDFPreprocess()

(16/16)

In [15]:
pd.set_option('display.max_columns', forScoringDF.shape[1])
forScoringDF.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 24 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   枠番               16 non-null     float64 
 1   馬番               16 non-null     float64 
 2   斤量               16 non-null     float64 
 3   単勝               16 non-null     float64 
 4   人気               16 non-null     int64   
 5   course_len       16 non-null     int32   
 6   weather          16 non-null     category
 7   raceType         16 non-null     category
 8   groundState      16 non-null     category
 9   Sex              16 non-null     category
 10  Age              16 non-null     int32   
 11  Weight           16 non-null     float64 
 12  WeightVariation  16 non-null     float64 
 13  pre1Rank         16 non-null     float64 
 14  pre2Rank         16 non-null     float64 
 15  pre3Rank         16 non-null     float64 
 16  pre4Rank         16 non-null     float64 
 17 

# スコアリング

## lightGBM

In [16]:
with open("../models/lightGBM/LGBM_1.pickle", "rb") as f:
        modelLGBM = pickle.load(f)

socredDataLGBM = modelLGBM.predict(forScoringDF)
raceCardDF["pred"] = socredDataLGBM

In [18]:
pd.set_option('display.max_columns', raceCardDF.shape[1])
raceCardDF.sort_values(by="pred", ascending=False)

,枠,馬番,印,馬名,性齢,斤量,騎手,厩舎,馬体重(増減),予想オッズ,人気,登録,グループ,馬メモ切替,raceID,horseID,jockeyID,raceName,course_len,weather,raceType,groundState,date,Sex,Age,Weight,WeightVariation,pred
2,NaN,NaN,--,グリフォン,牝3,55.0,鮫島駿,栗東小椋,NaN,5.0,2,NaN,NaN,編集,202502010301,2022103899,01157,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.444821
11,NaN,NaN,--,フリアフロリダ,牝3,53.0,北村友,栗東小栗,NaN,5.2,3,NaN,NaN,編集,202502010301,2022110101,01102,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.426476
7,NaN,NaN,--,ニシノコイブミ,牝3,55.0,武豊,美浦西田,NaN,4.5,1,NaN,NaN,編集,202502010301,2022100400,00666,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.382361
14,NaN,NaN,--,ロネット,牝3,53.0,△長浜,栗東浜田,NaN,8.5,4,NaN,NaN,編集,202502010301,2022101459,01214,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.295037
1,NaN,NaN,--,グッバイウェーブ,牝3,55.0,横山武,美浦中川,NaN,8.8,5,NaN,NaN,編集,202502010301,2022102492,01170,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.271907
4,NaN,NaN,--,ゴールドサニーデイ,牝3,55.0,藤岡佑,栗東渡辺,NaN,10.2,6,NaN,NaN,編集,202502010301,2022105330,01093,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.266939
5,NaN,NaN,--,シルフラ,牝3,55.0,佐々木,美浦大竹,NaN,13.9,8,NaN,NaN,編集,202502010301,2022107077,01197,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.184481
10,NaN,NaN,--,ブラックティンカー,牝3,55.0,吉田隼,美浦和田雄,NaN,13.9,7,NaN,NaN,編集,202502010301,2022100075,01095,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.166566
13,NaN,NaN,--,ルブリアン,牡3,54.0,▲鷲頭,栗東佐々木,NaN,23.5,9,NaN,NaN,編集,202502010301,2022106320,01202,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牡,3,NaN,NaN,0.128002
6,NaN,NaN,--,デルタニュートラル,牝3,52.0,▲上里,美浦加藤士,NaN,24.0,10,NaN,NaN,編集,202502010301,2022107243,01217,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.048144


## NN

In [20]:
forScoring_x = forScoringDF.values
forScoring_x = forScoring_x.astype(np.float32)

In [21]:
class Mychain(chainer.Chain):
    def __init__(self):
        super().__init__(
            bn = L.BatchNormalization(24),
            l1 = L.Linear(None, 24),
            l2 = L.Linear(None, 12),
            l3 = L.Linear(None, 6),
            l4 = L.Linear(None, 2),
        )

    def __call__(self,x):
        h1 = F.relu(self.l1(self.bn(x)))
        h2 = F.relu(self.l2(h1))
        h3 = F.relu(self.l3(h2))
        output = self.l4(h3)

        return output
    
model = L.Classifier(Mychain())
serializers.load_npz("../models/neuralNetwork/NN_1.npz",model)

In [22]:
socredDataNN = F.softmax(model.predictor(forScoring_x).data)[:,1].array

In [23]:
raceCardDF["predNN"] = socredDataNN

In [24]:
raceCardDF.sort_values(by="predNN", ascending=False)

,枠,馬番,印,馬名,性齢,斤量,騎手,厩舎,馬体重(増減),予想オッズ,人気,登録,グループ,馬メモ切替,...,horseID,jockeyID,raceName,course_len,weather,raceType,groundState,date,Sex,Age,Weight,WeightVariation,pred,predNN
7,NaN,NaN,--,ニシノコイブミ,牝3,55.0,武豊,美浦西田,NaN,4.5,1,NaN,NaN,編集,...,2022100400,00666,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.382361,0.813270
2,NaN,NaN,--,グリフォン,牝3,55.0,鮫島駿,栗東小椋,NaN,5.0,2,NaN,NaN,編集,...,2022103899,01157,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.444821,0.781645
11,NaN,NaN,--,フリアフロリダ,牝3,53.0,北村友,栗東小栗,NaN,5.2,3,NaN,NaN,編集,...,2022110101,01102,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.426476,0.744858
14,NaN,NaN,--,ロネット,牝3,53.0,△長浜,栗東浜田,NaN,8.5,4,NaN,NaN,編集,...,2022101459,01214,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.295037,0.600140
1,NaN,NaN,--,グッバイウェーブ,牝3,55.0,横山武,美浦中川,NaN,8.8,5,NaN,NaN,編集,...,2022102492,01170,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.271907,0.505523
4,NaN,NaN,--,ゴールドサニーデイ,牝3,55.0,藤岡佑,栗東渡辺,NaN,10.2,6,NaN,NaN,編集,...,2022105330,01093,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.266939,0.387290
10,NaN,NaN,--,ブラックティンカー,牝3,55.0,吉田隼,美浦和田雄,NaN,13.9,7,NaN,NaN,編集,...,2022100075,01095,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.166566,0.298912
5,NaN,NaN,--,シルフラ,牝3,55.0,佐々木,美浦大竹,NaN,13.9,8,NaN,NaN,編集,...,2022107077,01197,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.184481,0.255137
13,NaN,NaN,--,ルブリアン,牡3,54.0,▲鷲頭,栗東佐々木,NaN,23.5,9,NaN,NaN,編集,...,2022106320,01202,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牡,3,NaN,NaN,0.128002,0.172989
6,NaN,NaN,--,デルタニュートラル,牝3,52.0,▲上里,美浦加藤士,NaN,24.0,10,NaN,NaN,編集,...,2022107243,01217,3歳未勝利,1200,NaN,芝,NaN,2025-06-21,牝,3,NaN,NaN,0.048144,0.130368


# 全実行

In [7]:
tgtyear = 2025
tgtmonth = 6
tgtday = 21

In [8]:
tgtdate = datetime.date(tgtyear, tgtmonth, tgtday)  # 対象日付を指定
raceListURL = makeRaceDateURLList(tgtdate)
raceALLDF = makeRaceURLList(raceListURL)

with open("../models/lightGBM/LGBM_1.pickle", "rb") as f:
        modelLGBM = pickle.load(f)

class Mychain(chainer.Chain):
    def __init__(self):
        super().__init__(
            bn = L.BatchNormalization(24),
            l1 = L.Linear(None, 24),
            l2 = L.Linear(None, 12),
            l3 = L.Linear(None, 6),
            l4 = L.Linear(None, 2),
        )

    def __call__(self,x):
        h1 = F.relu(self.l1(self.bn(x)))
        h2 = F.relu(self.l2(h1))
        h3 = F.relu(self.l3(h2))
        output = self.l4(h3)

        return output
    
model = L.Classifier(Mychain())
serializers.load_npz("../models/neuralNetwork/NN_1.npz",model)

対象日付: 20250621
レース数: 36


In [47]:
raceID = screpingRaceCardHTML(raceALLDF["URL"][0])

raceCardDF = makeRaceCardDataFrame(raceID)

#raceCardDFのhorseIDのデータを作成
from makeRawData import makeHorseData
a = makeHorseData(raceCardDF["horseID"].tolist())
a.getHorseHTML(skipResult=False)
a.makeHorseRawData()

#スコアリング用データの作成
a = preprocessing(raceCardDF)
a.raceCardPreprocess()
a.makedataHorseResults()
a.horseResultsPreprocess()
a.makeVarFromHorseResults()
forScoringDF = a.forScoringDFPreprocess()

#lightGBMによるスコアリング
socredDataLGBM = modelLGBM.predict(forScoringDF)
raceCardDF["predLGBM"] = socredDataLGBM

#NNによるスコアリング
forScoring_x = forScoringDF.values
forScoring_x = forScoring_x.astype(np.float32)
socredDataNN = F.softmax(model.predictor(forScoring_x).data)[:,1].array
raceCardDF["predNN"] = socredDataNN

[raceID:202502010301]出馬表のHTMLをスクレイピングしました。
(16/16)seID:2022107077] の過去戦績のrawデータを作成中(16/16)

In [48]:
pd.set_option('display.max_columns', raceCardDF.shape[1])
raceCardDF

,枠,馬番,印,馬名,性齢,斤量,騎手,厩舎,馬体重(増減),オッズ更新,人気,登録,グループ,馬メモ切替,raceID,horseID,jockeyID,raceName,course_len,weather,raceType,groundState,date,Sex,Age,Weight,WeightVariation,predLGBM,predNN
0,1,1,--,フリアフロリダ,牝3,53.0,北村友,栗東小栗,NaN,5.8,3,NaN,NaN,編集,202502010301,2022110101,01102,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.382244,0.741166
1,1,2,--,ブラックティンカー,牝3,55.0,吉田隼,美浦和田雄,NaN,4.2,1,NaN,NaN,編集,202502010301,2022100075,01095,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.378290,0.794874
2,2,3,--,クレドール,牝3,55.0,原田和,美浦小笠,NaN,120.3,16,NaN,NaN,編集,202502010301,2022105653,01143,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.020629,0.012278
3,2,4,--,ファリア,牡3,54.0,▲古川奈,美浦土田,NaN,73.8,12,NaN,NaN,編集,202502010301,2022106894,01190,3歳未勝利,1200,晴,芝,良,2025-06-21,牡,3,NaN,NaN,0.041319,0.030436
4,3,5,--,ゴールドサニーデイ,牝3,55.0,藤岡佑,栗東渡辺,NaN,6.8,4,NaN,NaN,編集,202502010301,2022105330,01093,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.363358,0.573600
5,3,6,--,フィリグラン,牝3,55.0,原,栗東森田,NaN,82.3,13,NaN,NaN,編集,202502010301,2022100644,01184,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.025906,0.021786
6,4,7,--,ワタシマツワ,牝3,51.0,★小林美,美浦竹内,NaN,32.3,8,NaN,NaN,編集,202502010301,2022101412,01206,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.082578,0.104390
7,4,8,--,デルタニュートラル,牝3,52.0,▲上里,美浦加藤士,NaN,49.8,10,NaN,NaN,編集,202502010301,2022107243,01217,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.030408,0.050624
8,5,9,--,ロネット,牝3,53.0,△長浜,栗東浜田,NaN,8.1,5,NaN,NaN,編集,202502010301,2022101459,01214,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.287581,0.590929
9,5,10,--,ルブリアン,牡3,54.0,▲鷲頭,栗東佐々木,NaN,66.3,11,NaN,NaN,編集,202502010301,2022106320,01202,3歳未勝利,1200,晴,芝,良,2025-06-21,牡,3,NaN,NaN,0.039128,0.055346


In [25]:
import schedule 

In [42]:
a = datetime.datetime(2025,6,19,22,26,00)
b = datetime.datetime(2025,6,19,22,27,00)

In [43]:
timeList = [a,b]

In [44]:
timeList

[datetime.datetime(2025, 6, 19, 22, 26),
 datetime.datetime(2025, 6, 19, 22, 27)]

In [32]:
a = datetime.datetime.now()

In [41]:
a.minute

20

In [ ]:
for tgttime in runtimeList:
    returnflg = 0

    while returnflg == 0:
        now = datetime.datetime.now()
        if now.hour == tgttime.hour and now.minute == tgttime.minute :
            print(f"現在時刻は{now.hour}:{now.minute}です")
            returnflg = 1
        else :
            time.sleep(10)


現在時刻は22:26です
現在時刻は22:27です


In [94]:
runtimeList = []
for tgt in raceALLDF["datetime"]:
    runtime = tgt + datetime.timedelta(minutes=-5)
    runtimeList.append(runtime)

In [ ]:
for tgttime in runtimeList:
    returnflg = 0

    while returnflg == 0:
        now = datetime.datetime.now()
        if now.hour == tgttime.hour and now.minute == tgttime.minute :
            print(f"現在時刻は{now.hour}:{now.minute}です")
            returnflg = 1
        else :
            time.sleep(10)


In [39]:
from linebot import LineBotApi
from linebot.v3.messaging import Configuration, ApiClient, MessagingApi, PushMessageRequest, TextMessage

CHANNEL_ACCESS_TOKEN = "iFSTH9Vagd7Ozy2YcYiH+xrjx+rmQL8xDWjZtAnIaT78vhIPhDdYQ0O7a1CSpuH2VxYeifzWNAdJWEgAkSHf2VNH9woaYVm2tB9HgWCFTRmfFZtDaJmTcUbZX0whj6/uQWUp9dquBfef24HYaWzejgdB04t89/1O/w1cDnyilFU="
USER_ID = "U176770efe9b2a6d6bae891eb95ca0f45"

configuration = Configuration(access_token=CHANNEL_ACCESS_TOKEN)
with ApiClient(configuration) as api_client:
   messaging_api = MessagingApi(api_client)
   message = TextMessage(text=text)
   push_message_request = PushMessageRequest(to=USER_ID,messages=[message])
   messaging_api.push_message(push_message_request)

YYYY+競馬場番号(00) + 開催回数(00) + 開催日(00) + レース番号(00)
0123 45              67            89           1011

In [ ]:
01:札幌,02:函館,03:福島,04:新潟,05:東京,06:中山,07:中京,08:京都,09:阪神,10:小倉  

In [29]:
raceCardDF

,枠,馬番,印,馬名,性齢,斤量,騎手,厩舎,馬体重(増減),オッズ更新,人気,登録,グループ,馬メモ切替,raceID,horseID,jockeyID,raceName,course_len,weather,raceType,groundState,date,Sex,Age,Weight,WeightVariation,pred,predNN
0,1,1,--,フリアフロリダ,牝3,53.0,北村友,栗東小栗,NaN,6.4,4,NaN,NaN,編集,202502010301,2022110101,01102,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.360098,0.670736
1,1,2,--,ブラックティンカー,牝3,55.0,吉田隼,美浦和田雄,NaN,4.5,2,NaN,NaN,編集,202502010301,2022100075,01095,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.407632,0.718805
2,2,3,--,クレドール,牝3,55.0,原田和,美浦小笠,NaN,183.4,16,NaN,NaN,編集,202502010301,2022105653,01143,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.009994,0.007424
3,2,4,--,ファリア,牡3,54.0,▲古川奈,美浦土田,NaN,79.1,11,NaN,NaN,編集,202502010301,2022106894,01190,3歳未勝利,1200,晴,芝,良,2025-06-21,牡,3,NaN,NaN,0.041319,0.034941
4,3,5,--,ゴールドサニーデイ,牝3,55.0,藤岡佑,栗東渡辺,NaN,6.1,3,NaN,NaN,編集,202502010301,2022105330,01093,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.393017,0.596002
5,3,6,--,フィリグラン,牝3,55.0,原,栗東森田,NaN,91.7,14,NaN,NaN,編集,202502010301,2022100644,01184,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.023102,0.023262
6,4,7,--,ワタシマツワ,牝3,51.0,★小林美,美浦竹内,NaN,42.9,9,NaN,NaN,編集,202502010301,2022101412,01206,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.063231,0.078831
7,4,8,--,デルタニュートラル,牝3,52.0,▲上里,美浦加藤士,NaN,59.3,10,NaN,NaN,編集,202502010301,2022107243,01217,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.024444,0.049385
8,5,9,--,ロネット,牝3,53.0,△長浜,栗東浜田,NaN,7.6,5,NaN,NaN,編集,202502010301,2022101459,01214,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.299312,0.559075
9,5,10,--,ルブリアン,牡3,54.0,▲鷲頭,栗東佐々木,NaN,89.6,13,NaN,NaN,編集,202502010301,2022106320,01202,3歳未勝利,1200,晴,芝,良,2025-06-21,牡,3,NaN,NaN,0.032025,0.037933


In [76]:
raceID

'202502010301'

In [78]:
kaisu

'02'

In [80]:
kaisu = raceID[6:8]

In [82]:
kaisu = raceID[6:8]
kaisaibi = raceID[8:10]

if raceID[4:6] == "01":
    trackName = "札幌"
elif raceID[4:6] == "02":
    trackName = "函館"
elif raceID[4:6] == "03":
    trackName = "福島"
elif raceID[4:6] == "04":
    trackName = "新潟"
elif raceID[4:6] == "05":
    trackName = "東京"
elif raceID[4:6] == "06":
    trackName = "中山"
elif raceID[4:6] == "07":
    trackName = "中京"
elif raceID[4:6] == "08":
    trackName = "京都"
elif raceID[4:6] == "09":
    trackName = "阪神"
elif raceID[4:6] == "10":
    trackName = "小倉"

raceNo = raceID[10:]
raceName = raceCardDF["raceName"][0]

raceType = raceCardDF["raceType"][0]
course_len = raceCardDF["course_len"][0]
weather = raceCardDF["weather"][0]
groundState = raceCardDF["groundState"][0]

In [84]:
raceCardDF.sort_values(by="predLGBM", ascending=False,inplace=True)
raceCardDF.reset_index(inplace=True)
LGBM1st = raceCardDF["馬名"][0] + "/" + str(raceCardDF["オッズ更新"][0]) + "/" + str(raceCardDF["predLGBM"][0])
LGBM2nd = raceCardDF["馬名"][1] + "/" + str(raceCardDF["オッズ更新"][1]) + "/" + str(raceCardDF["predLGBM"][1])
LGBM3rd = raceCardDF["馬名"][2] + "/" + str(raceCardDF["オッズ更新"][2]) + "/" + str(raceCardDF["predLGBM"][2])

raceCardDF.sort_values(by="predNN", ascending=False,inplace=True)
raceCardDF.reset_index(inplace=True)
NN1st = raceCardDF["馬名"][0] + "/" + str(raceCardDF["オッズ更新"][0]) + "/" + str(raceCardDF["predNN"][0])
NN2nd = raceCardDF["馬名"][1] + "/" + str(raceCardDF["オッズ更新"][1]) + "/" + str(raceCardDF["predNN"][1])
NN3rd = raceCardDF["馬名"][2] + "/" + str(raceCardDF["オッズ更新"][2]) + "/" + str(raceCardDF["predNN"][2])

ValueError: cannot insert level_0, already exists

In [73]:
text = f"{int(kaisu)}回 {trackName} {int(kaisaibi)}日目" + "\n" + f"{int(raceNo)}R {raceName}" + "\n" + f"{raceType}{course_len} 天気:{weather} 馬場:{groundState}" + "\n\n" +"【lightGBM予測結果】馬名/単勝オッズ/予測値" + "\n" + LGBM1st  + "\n" + LGBM2nd  + "\n" + LGBM3rd

In [74]:
print(text)

2回 函館 3日目
1R 3歳未勝利
芝1200 天気:晴 馬場:良

【lightGBM予測結果】馬名/単勝オッズ/予測値
ニシノコイブミ/4.2/0.4332083282885192
フリアフロリダ/5.8/0.38224357394740427
ブラックティンカー/4.2/0.37828963255466425


In [70]:
raceCardDF.sort_values(by="predLGBM", ascending=False,inplace=True)
raceCardDF.reset_index(inplace=True)

In [71]:
raceCardDF

,index,枠,馬番,印,馬名,性齢,斤量,騎手,厩舎,馬体重(増減),オッズ更新,人気,登録,グループ,...,horseID,jockeyID,raceName,course_len,weather,raceType,groundState,date,Sex,Age,Weight,WeightVariation,predLGBM,predNN
0,14,8,15,--,ニシノコイブミ,牝3,55.0,武豊,美浦西田,NaN,4.2,2,NaN,NaN,...,2022100400,00666,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.433208,0.806339
1,0,1,1,--,フリアフロリダ,牝3,53.0,北村友,栗東小栗,NaN,5.8,3,NaN,NaN,...,2022110101,01102,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.382244,0.741166
2,1,1,2,--,ブラックティンカー,牝3,55.0,吉田隼,美浦和田雄,NaN,4.2,1,NaN,NaN,...,2022100075,01095,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.378290,0.794874
3,4,3,5,--,ゴールドサニーデイ,牝3,55.0,藤岡佑,栗東渡辺,NaN,6.8,4,NaN,NaN,...,2022105330,01093,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.363358,0.573600
4,8,5,9,--,ロネット,牝3,53.0,△長浜,栗東浜田,NaN,8.1,5,NaN,NaN,...,2022101459,01214,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.287581,0.590929
5,11,6,12,--,グリフォン,牝3,55.0,鮫島駿,栗東小椋,NaN,9.6,6,NaN,NaN,...,2022103899,01157,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.271952,0.565249
6,10,6,11,--,グッバイウェーブ,牝3,55.0,横山武,美浦中川,NaN,10.4,7,NaN,NaN,...,2022102492,01170,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.234831,0.423226
7,15,8,16,--,シルフラ,牝3,55.0,佐々木,美浦大竹,NaN,41.6,9,NaN,NaN,...,2022107077,01197,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.101562,0.114885
8,6,4,7,--,ワタシマツワ,牝3,51.0,★小林美,美浦竹内,NaN,32.3,8,NaN,NaN,...,2022101412,01206,3歳未勝利,1200,晴,芝,良,2025-06-21,牝,3,NaN,NaN,0.082578,0.104390
9,3,2,4,--,ファリア,牡3,54.0,▲古川奈,美浦土田,NaN,73.8,12,NaN,NaN,...,2022106894,01190,3歳未勝利,1200,晴,芝,良,2025-06-21,牡,3,NaN,NaN,0.041319,0.030436


In [67]:
print(LGBM1st)

フリアフロリダ/5.8/0.38224357394740427
